# Forecast Pipeline Automático

Executa sequencialmente todos os notebooks de forecasting usando **papermill**.
Cada notebook roda no seu kernel/ambiente Python correto.

**Ambientes:**
- `python3` (.venv, Python 3.10) — maioria dos notebooks
- `lag_llama_39` (venv39, Python 3.9) — LagLlama
- `venv_tempo` (venv_tempo, Python 3.9) — TEMPO

**Ordem de execução:**
1. Importação da base de dados
2. Análise Exploratória
3. Modelos estatísticos: ARIMA, SARIMAX, ETS, Theta, ARCH, GARCH
4. Machine Learning: XGBoost, RandomForest, Prophet
5. Foundation Models: Chronos, LagLlama, TimesFM, TEMPO
6. Comparação consolidada dos modelos

In [15]:
import papermill as pm
import os
import time
from pathlib import Path
from datetime import datetime

In [16]:
NOTEBOOKS_DIR = Path("notebooks")
OUTPUT_DIR = NOTEBOOKS_DIR / "executed"
OUTPUT_DIR.mkdir(exist_ok=True)

# Kernel padrão (.venv Python 3.10)
DEFAULT_KERNEL = "python3"

# Mapeamento: notebook -> kernel específico
# Notebooks não listados aqui usam DEFAULT_KERNEL
KERNEL_MAP = {
    "Forecasting_LagLlama.ipynb":  "lag_llama_39",   # Python 3.9 (venv39)
    "Forecasting_TEMPO.ipynb":     "venv_tempo",     # Python 3.9 (venv_tempo)
}

# Lista ordenada dos notebooks de forecast
NOTEBOOKS = [
# # 1. Importação da base
    "Forecast_busca_bases_macroeconomia.ipynb",
# # 2. Análise Exploratória
    "Forecasting_analise_exploratoria.ipynb",
# # 3. Modelos Estatísticos
    "Forecasting_ARIMA.ipynb",
    "Forecasting_SARIMAX.ipynb",
    "Forecasting_ETS.ipynb",
    "Forecasting_Theta.ipynb",
    "Forecasting_ARCH.ipynb",
    "Forecasting_GARCH.ipynb",
# # 4. Machine Learning
    "Forecasting_XGBoost.ipynb",
    "Forecasting_RandomForest.ipynb",
    "Forecasting_Prophet.ipynb",
# # 5. Foundation Models
    "Forecasting_Chronos.ipynb",
    "Forecasting_LagLlama.ipynb",
    "Forecasting_TimesFM.ipynb",
    "Forecasting_TEMPO.ipynb",
# # 6. Comparação
    "Forecast_comparacao_modelo.ipynb",
]

kernels_usados = sorted(set(KERNEL_MAP.values()))
print(f"Total de notebooks a executar: {len(NOTEBOOKS)}")
print(f"Kernels disponíveis: {DEFAULT_KERNEL} (padrão), {', '.join(kernels_usados)}")
print()
for i, nb_name in enumerate(NOTEBOOKS, 1):
    exists = (NOTEBOOKS_DIR / nb_name).exists()
    kernel = KERNEL_MAP.get(nb_name, DEFAULT_KERNEL)
    status = "OK" if exists else "NAO ENCONTRADO"
    print(f"  {i:2d}. {nb_name:50s} [{status}]  kernel={kernel}")


Total de notebooks a executar: 16
Kernels disponíveis: python3 (padrão), lag_llama_39, venv_tempo

   1. Forecast_busca_bases_macroeconomia.ipynb           [OK]  kernel=python3
   2. Forecasting_analise_exploratoria.ipynb             [OK]  kernel=python3
   3. Forecasting_ARIMA.ipynb                            [OK]  kernel=python3
   4. Forecasting_SARIMAX.ipynb                          [OK]  kernel=python3
   5. Forecasting_ETS.ipynb                              [OK]  kernel=python3
   6. Forecasting_Theta.ipynb                            [OK]  kernel=python3
   7. Forecasting_ARCH.ipynb                             [OK]  kernel=python3
   8. Forecasting_GARCH.ipynb                            [OK]  kernel=python3
   9. Forecasting_XGBoost.ipynb                          [OK]  kernel=python3
  10. Forecasting_RandomForest.ipynb                     [OK]  kernel=python3
  11. Forecasting_Prophet.ipynb                          [OK]  kernel=python3
  12. Forecasting_Chronos.ipynb            

In [ ]:
results = []
pipeline_start = time.time()

for i, nb_name in enumerate(NOTEBOOKS, 1):
    input_path = NOTEBOOKS_DIR / nb_name
    output_path = OUTPUT_DIR / nb_name
    kernel = KERNEL_MAP.get(nb_name, DEFAULT_KERNEL)

    if not input_path.exists():
        print(f"[{i}/{len(NOTEBOOKS)}] PULANDO {nb_name} (arquivo não encontrado)")
        results.append({"notebook": nb_name, "kernel": kernel, "status": "SKIPPED", "duration": 0, "error": "File not found"})
        continue

    print(f"[{i}/{len(NOTEBOOKS)}] Executando {nb_name} (kernel={kernel})...", end=" ", flush=True)
    start = time.time()

    try:
        pm.execute_notebook(
            str(input_path),
            str(output_path),
            kernel_name=kernel,
            cwd=str(NOTEBOOKS_DIR),
        )
        elapsed = time.time() - start
        print(f"OK ({elapsed:.1f}s)")
        results.append({"notebook": nb_name, "kernel": kernel, "status": "OK", "duration": elapsed, "error": None})

    except pm.PapermillExecutionError as e:
        elapsed = time.time() - start
        print(f"ERRO ({elapsed:.1f}s)")
        print(f"   -> {e.ename}: {e.evalue}")
        results.append({"notebook": nb_name, "kernel": kernel, "status": "ERROR", "duration": elapsed, "error": f"{e.ename}: {e.evalue}"})

    except Exception as e:
        elapsed = time.time() - start
        print(f"ERRO ({elapsed:.1f}s)")
        print(f"   -> {type(e).__name__}: {e}")
        results.append({"notebook": nb_name, "kernel": kernel, "status": "ERROR", "duration": elapsed, "error": str(e)})

pipeline_elapsed = time.time() - pipeline_start
print(f"\n{'='*60}")
print(f"Pipeline finalizado em {pipeline_elapsed/60:.1f} min ({pipeline_elapsed:.0f}s)")

[1/16] Executando Forecast_busca_bases_macroeconomia.ipynb (kernel=python3)... 

Executing: 100%|██████████| 24/24 [01:13<00:00,  3.08s/cell]

OK (73.9s)
[2/16] Executando Forecasting_analise_exploratoria.ipynb (kernel=python3)... 


Executing: 100%|██████████| 22/22 [01:08<00:00,  3.11s/cell]

OK (68.6s)
[3/16] Executando Forecasting_ARIMA.ipynb (kernel=python3)... 


Executing: 100%|██████████| 22/22 [22:15<00:00, 60.71s/cell] 

OK (1335.7s)
[4/16] Executando Forecasting_SARIMAX.ipynb (kernel=python3)... 


Executing:   6%|▌         | 1/18 [00:03<00:52,  3.12s/cell]

In [ ]:
# Resumo dos resultados
total_duration = sum(r['duration'] for r in results)
horas, resto = divmod(total_duration, 3600)
minutos, segundos = divmod(resto, 60)

print(f"{'='*70}")
print(f"{'NOTEBOOK':<50s} {'STATUS':>7s} {'TEMPO':>8s}")
print(f"{'-'*70}")
for r in results:
    m, s = divmod(r['duration'], 60)
    tempo_str = f"{int(m)}m{int(s):02d}s" if m >= 1 else f"{r['duration']:.1f}s"
    print(f"{r['notebook']:<50s} {r['status']:>7s} {tempo_str:>8s}")
print(f"{'-'*70}")

print(f"\nTotal de notebooks: {len(results)}")
print(f"  Sucesso: {sum(1 for r in results if r['status']=='OK')}")
print(f"  Erro:    {sum(1 for r in results if r['status']=='ERROR')}")
print(f"  Pulados: {sum(1 for r in results if r['status']=='SKIPPED')}")

if horas >= 1:
    print(f"\nTempo total de execução: {int(horas)}h {int(minutos)}m {int(segundos)}s")
else:
    print(f"\nTempo total de execução: {int(minutos)}m {int(segundos)}s")

print(f"Tempo médio por notebook: {total_duration/max(len(results),1):.0f}s")
print(f"Finalizado em: {datetime.now():%Y-%m-%d %H:%M:%S}")

NOTEBOOK                                            STATUS    TEMPO
----------------------------------------------------------------------
Forecast_busca_bases_macroeconomia.ipynb                OK    1m06s
Forecasting_analise_exploratoria.ipynb                  OK    1m04s
Forecasting_ARIMA.ipynb                                 OK   19m08s
Forecasting_SARIMAX.ipynb                               OK    1m59s
Forecasting_ETS.ipynb                                   OK    38.8s
Forecasting_Theta.ipynb                                 OK    12.9s
Forecasting_ARCH.ipynb                                  OK    3m40s
Forecasting_GARCH.ipynb                                 OK    9m59s
Forecasting_XGBoost.ipynb                               OK    5m11s
Forecasting_RandomForest.ipynb                          OK    7m08s
Forecasting_Prophet.ipynb                               OK    3m47s
Forecasting_Chronos.ipynb                               OK    1m39s
Forecasting_LagLlama.ipynb                   

In [ ]:

# =====================================================================
# EXPORTAÇÃO DOS NOTEBOOKS EXECUTADOS PARA HTML/PDF
# =====================================================================
# Converte cada notebook executado para HTML com saídas completas.
# Abra os arquivos no browser e use Ctrl+P > Salvar como PDF.
# =====================================================================

import subprocess
import shutil
from pathlib import Path
from datetime import datetime

REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True)

executed_notebooks = sorted(OUTPUT_DIR.glob("*.ipynb"))
exportados = []
falhas = []

print(f"Exportando {len(executed_notebooks)} notebook(s) para HTML...\n")

for nb_path in executed_notebooks:
    out_html = REPORTS_DIR / (nb_path.stem + ".html")
    print(f"  {nb_path.name} ...", end=" ", flush=True)
    try:
        result = subprocess.run(
            [
                ".venv\\Scripts\\jupyter.exe", "nbconvert",
                "--to", "html",
                "--output", str(out_html.resolve()),
                str(nb_path.resolve()),
            ],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode == 0 and out_html.exists():
            size_kb = out_html.stat().st_size // 1024
            print(f"OK ({size_kb} KB)")
            exportados.append(out_html)
        else:
            print(f"FALHA")
            falhas.append((nb_path.name, result.stderr[-300:] if result.stderr else ""))
    except Exception as e:
        print(f"ERRO: {e}")
        falhas.append((nb_path.name, str(e)))

# --- Relatório HTML consolidado do pipeline ---
ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
summary_html = REPORTS_DIR / "00_pipeline_summary.html"

status_rows = ""
for r in results:
    cor = "#d4edda" if r["status"] == "OK" else ("#fff3cd" if r["status"] == "SKIPPED" else "#f8d7da")
    m, s = divmod(r["duration"], 60)
    tempo = f"{int(m)}m{int(s):02d}s" if m >= 1 else f"{r['duration']:.1f}s"
    erro = r["error"] or ""
    status_rows += f'<tr style="background:{cor}"><td>{r["notebook"]}</td><td><b>{r["status"]}</b></td><td>{tempo}</td><td style="font-size:0.85em;color:#555">{erro}</td></tr>\n'

links = "".join(
    f'<li><a href="{p.name}">{p.stem}</a></li>\n'
    for p in sorted(exportados)
)

html_content = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="utf-8">
  <title>Forecast Pipeline — Resumo {ts}</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 40px; color: #222; }}
    h1 {{ color: #1a1a2e; }}
    table {{ border-collapse: collapse; width: 100%; margin-top: 20px; }}
    th {{ background: #1a1a2e; color: white; padding: 8px 12px; text-align: left; }}
    td {{ padding: 7px 12px; border-bottom: 1px solid #ddd; }}
    ul {{ line-height: 2; }}
    a {{ color: #007bff; }}
  </style>
</head>
<body>
  <h1>Forecast Pipeline — Resumo de Execução</h1>
  <p><b>Data:</b> {ts} &nbsp;|&nbsp;
     <b>Sucesso:</b> {sum(1 for r in results if r['status']=='OK')} &nbsp;|&nbsp;
     <b>Erro:</b> {sum(1 for r in results if r['status']=='ERROR')} &nbsp;|&nbsp;
     <b>Pulados:</b> {sum(1 for r in results if r['status']=='SKIPPED')}
  </p>
  <table>
    <tr><th>Notebook</th><th>Status</th><th>Tempo</th><th>Erro</th></tr>
    {status_rows}
  </table>
  <h2 style="margin-top:40px">Notebooks Exportados</h2>
  <ul>{links}</ul>
  <p style="color:#888;font-size:0.85em">
    Para gerar PDF: abra cada link acima no browser e use <b>Ctrl+P → Salvar como PDF</b>.
  </p>
</body>
</html>"""

summary_html.write_text(html_content, encoding="utf-8")

print(f"\n{'='*60}")
print(f"Exportados com sucesso: {len(exportados)}/{len(executed_notebooks)}")
if falhas:
    print(f"Falhas: {len(falhas)}")
    for nome, msg in falhas:
        print(f"  - {nome}: {msg[:120]}")
print(f"\nRelatórios salvos em: {REPORTS_DIR.resolve()}")
print(f"  -> Resumo do pipeline: {summary_html.name}")
for p in sorted(exportados):
    print(f"  -> {p.name}")
print(f"\nAbra os arquivos .html no browser e use Ctrl+P → Salvar como PDF.")


Exportando 16 notebook(s) para HTML...

  Forecast_busca_bases_macroeconomia.ipynb ... OK (862 KB)
  Forecast_comparacao_modelo.ipynb ... OK (1572 KB)
  Forecasting_analise_exploratoria.ipynb ... OK (7373 KB)
  Forecasting_ARCH.ipynb ... OK (438 KB)
  Forecasting_ARIMA.ipynb ... OK (649 KB)
  Forecasting_Chronos.ipynb ... OK (641 KB)
  Forecasting_ETS.ipynb ... OK (635 KB)
  Forecasting_GARCH.ipynb ... OK (470 KB)
  Forecasting_LagLlama.ipynb ... OK (658 KB)
  Forecasting_Prophet.ipynb ... OK (752 KB)
  Forecasting_RandomForest.ipynb ... OK (684 KB)
  Forecasting_SARIMAX.ipynb ... OK (651 KB)
  Forecasting_TEMPO.ipynb ... OK (652 KB)
  Forecasting_Theta.ipynb ... OK (644 KB)
  Forecasting_TimesFM.ipynb ... OK (658 KB)
  Forecasting_XGBoost.ipynb ... OK (657 KB)

Exportados com sucesso: 16/16

Relatórios salvos em: C:\Users\phill\Projetos\lag-llama\reports
  -> Resumo do pipeline: 00_pipeline_summary.html
  -> Forecast_busca_bases_macroeconomia.html
  -> Forecast_comparacao_modelo.html
